In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Charger le fichier de données brut
df = pd.read_csv('../data/raw/diabetes/diabetes.csv')

# 2. Nettoyage des données (remplacement des 0 aberrants par la médiane)
columns_with_zeros = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
df[columns_with_zeros] = df[columns_with_zeros].replace(0, np.nan)

for col in columns_with_zeros:
    median_val = df[col].median()
    df[col].fillna(median_val, inplace=True)

print("Nettoyage des données effectué avec succès.")

# 3. Séparation des features (X) et de la target (y)
target_column = 'Outcome' if 'Outcome' in df.columns else df.columns[-1]
X = df.drop(columns=[target_column])
y = df[target_column]

# 4. Division en train/test avec stratification (essentiel pour garder la proportion des classes)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Dimensions de X_train : {X_train.shape} | X_test : {X_test.shape}")

Nettoyage des données effectué avec succès.
Dimensions de X_train : (614, 8) | X_test : (154, 8)


/tmp/ipykernel_164120/2015100138.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(median_val, inplace=True)
/tmp/ipykernel_164120/2015100138.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try us

In [2]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib

def optimize_random_forest(X_train, y_train, X_test, y_test):
    param_dist = {
        'n_estimators': [50, 100, 200, 300, 500],
        'max_depth': [None, 10, 20, 30, 40],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'bootstrap': [True, False],
        'class_weight': ['balanced', 'balanced_subsample', None]
    }

    rf = RandomForestClassifier(random_state=42)

    random_search = RandomizedSearchCV(
        estimator=rf,
        param_distributions=param_dist,
        n_iter=20,
        cv=5,
        scoring='f1',
        random_state=42,
        n_jobs=-1,
        verbose=1
    )

    print("Recherche des meilleurs hyperparamètres en cours...")
    random_search.fit(X_train, y_train)

    best_rf = random_search.best_estimator_
    print(f"\nMeilleurs paramètres trouvés :\n{random_search.best_params_}")

    y_pred = best_rf.predict(X_test)
    
    print("\n--- Évaluation du Modèle Optimisé ---")
    print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
    print("\nMatrice de confusion :")
    print(confusion_matrix(y_test, y_pred))
    print("\nRapport de classification :")
    print(classification_report(y_test, y_pred))

    # Sauvegarde du modèle optimisé
    joblib.dump(best_rf, "../models/diabetes_random_forest_optimized.pkl")
    print("\nModèle optimisé sauvegardé avec succès dans models/ !")

    return best_rf

In [3]:
best_rf_model = optimize_random_forest(X_train, y_train, X_test, y_test)

Recherche des meilleurs hyperparamètres en cours...
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Meilleurs paramètres trouvés :
{'n_estimators': 300, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_depth': None, 'class_weight': 'balanced', 'bootstrap': True}

--- Évaluation du Modèle Optimisé ---
Accuracy : 0.7532

Matrice de confusion :
[[79 21]
 [17 37]]

Rapport de classification :
              precision    recall  f1-score   support

           0       0.82      0.79      0.81       100
           1       0.64      0.69      0.66        54

    accuracy                           0.75       154
   macro avg       0.73      0.74      0.73       154
weighted avg       0.76      0.75      0.76       154


Modèle optimisé sauvegardé avec succès dans models/ !
